# Fondos de inversión tradicionales (no ETF)

Ejecuta todas las celdas. Este notebook descarga/carga, limpia y actualiza los CSV
que alimentan `04_conclusiones.ipynb`. Configuración común en `config.toml`.

**El ISIN es la identidad del activo.** Solo se estudian ISIN del catálogo cargado de
Trade Republic. Si el catálogo o la cobertura de precios son parciales se indica expresamente.
Por defecto se seleccionan **hasta 250 por tipo**, usando el orden predeterminado del
explorador TR como proxy de relevancia (fórmula no publicada), no el orden alfabético del ISIN.
La selección se guarda en `data/processed/mutual_fund/selection.csv`. El top por rentabilidad
se calcula después sobre esta selección. El máximo común está en `config.toml`.

Precio: último cierre o valor liquidativo, no cotización intradía. Se conserva el cierre
y su precio ajustado por dividendos/distribuciones y splits. Retorno logarítmico:
`ln(precio_ajustado_hoy / precio_ajustado_anterior)`.

Se elimina el duplicado de fecha conservando la última fila y se auditan los conflictos.
Se rellenan huecos de lunes a viernes con el dato anterior, incluidos festivos, sin añadir fines de
semana ni extender un activo más allá de su último dato. No se rellena hacia atrás.
Un exceso de huecos o un precio antiguo excluyen el activo del ranking.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path(os.environ["TR_PIPELINE_ROOT"]).resolve() if os.getenv("TR_PIPELINE_ROOT") else Path.cwd()
if not (ROOT / "config.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from tr_pipeline.pipeline import run_asset_type, conclusions
from tr_pipeline.notebook import show_coverage, show_eda, show_rankings
from IPython.display import display, Markdown
import pandas as pd

REFRESH = False  # True fuerza una descarga nueva de Yahoo y divisas.
OFFLINE = os.getenv("TR_PIPELINE_OFFLINE", "0") == "1"


## 1. Pipeline y cobertura

Una correspondencia dudosa entre ISIN y ticker queda pendiente para revisión.

In [ ]:
RESULT = run_asset_type(ROOT, "mutual_fund", refresh=REFRESH, offline=OFFLINE)
show_coverage(RESULT)

## 2. Limpieza, EDA y medias

Medias de los últimos 3, 6 y 12 meses naturales. Comparación en EUR, usando la misma serie ajustada. Los huecos imputados se conservan con una marca para auditar su efecto.

In [ ]:
show_eda(ROOT, "mutual_fund", RESULT)

## 3. Qué significa «sector» en un fondo

Se usa la categoría publicada del fondo: renta variable global, bonos, tecnología, etc.
Un fondo diversificado no se fuerza a pertenecer a un solo sector corporativo.
Solo si falta esa clasificación se agrupan trayectorias mediante clustering (mínimo
4 activos y 60 observaciones comunes). El resultado se llama «grupo estadístico»,
sin atribuirle un sector económico inventado. Las categorías conocidas se conservan.


## 4. Top 5 global y por sector/categoría

Orden: rentabilidad total de los últimos **12 meses en EUR**, con idéntica fecha de referencia.
Si hay menos de cinco activos elegibles se muestran los que existan. Acciones, fondos y ETF
se clasifican por separado. Los activos sin datos suficientes siguen visibles en el informe de cobertura.


In [ ]:
show_rankings(RESULT)